# Day 5：幻觉检测与安全风险

🔵 LLM 核心原理与安全基础 · 第 1 周

[在 GitHub 查看教程](https://github.com/Siebelyk/fde-daily-plan/blob/main/tutorials/Day-05.md)

## 学习目标

1. 理解 LLM 幻觉 (Hallucination) 的成因
2. 理解幻觉对安全的影响：错误信息、误导性输出
3. 实现一个简单的幻觉检测器


## 推荐资料

- 📄 论文 [Survey of Hallucination in LLMs](https://arxiv.org/abs/2309.01219)
- 🎬 视频 [Google - Hallucination in LLMs Explained](https://www.youtube.com/watch?v=QGM1gNZYbc8)
- 🔧 工具 [SelfCheckGPT](https://github.com/potsawee/selfcheckgpt)


## Demo：幻觉检测器：基于自一致性检测幻觉

通过多次采样同一 prompt，比较输出一致性来检测幻觉。高一致性 = 低幻觉概率

难度：基础 | 预计：2h

## 环境准备


In [ ]:
!pip install openai numpy matplotlib




## 原理速览
幻觉 = LLM 生成看似合理但实际错误的内容。成因：训练数据噪声、知识截止日期、过度自信。

自一致性检测原理：对同一 prompt 多次采样（temperature>0），如果输出高度一致，说明模型"确信"；
如果输出差异大，说明模型不确定 -> 可能是幻觉。

## 代码


In [ ]:
from openai import OpenAI
import numpy as np
from collections import Counter

client = OpenAI()

def sample_response(prompt, n=5, temp=0.7):
    """对同一 prompt 采样 n 次"""
    responses = []
    for _ in range(n):
        resp = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=100,
        )
        responses.append(resp.choices[0].message.content.strip())
    return responses

def self_consistency_score(responses):
    """计算自一致性分数（0-1，越高越一致）"""
    # 简化：用首句的 Jaccard 相似度
    def jaccard(a, b):
        sa, sb = set(a.lower().split()), set(b.lower().split())
        return len(sa & sb) / len(sa | sb) if sa | sb else 0

    scores = []
    for i in range(len(responses)):
        for j in range(i+1, len(responses)):
            scores.append(jaccard(responses[i], responses[j]))
    return np.mean(scores) if scores else 0

# 测试用例
test_cases = [
    {"type": "factual", "prompt": "中国首都是哪里？"},
    {"type": "ambiguous", "prompt": "2026年诺贝尔物理学奖得主是谁？"},  # 未来事件，可能幻觉
    {"type": "opinion", "prompt": "Python 和 Java 哪个更好？"},
]

for tc in test_cases:
    print(f"
=== {tc['type'].upper()} ===")
    print(f"Prompt: {tc['prompt']}")
    responses = sample_response(tc["prompt"], n=5)
    score = self_consistency_score(responses)
    for i, r in enumerate(responses):
        print(f"  Sample {i+1}: {r[:60]}...")
    print(f"  Consistency score: {score:.2f}")
    if score < 0.3:
        print(f"  [WARNING] Low consistency - possible hallucination!")
    elif score < 0.6:
        print(f"  [CAUTION] Moderate uncertainty")
    else:
        print(f"  [OK] High consistency")




## 安全分析
幻觉在安全场景下可能导致：错误的安全建议、伪造的漏洞信息、误导性的分析结论。检测方案：自一致性、外部知识验证、置信度评估。

## 进阶挑战

1. 尝试用 embedding 相似度替代 Jaccard，看效果是否更好
   - 思路提示：用 sentence-transformers 的 SentenceTransformer 编码后算 cosine similarity，对比 Jaccard 和 embedding 两种方法的一致性差异
   - 参考：[Sentence-Transformers 文档](https://www.sbert.net/)
2. 研究 SelfCheckGPT 的具体实现
   - 思路提示：SelfCheckGPT 通过多次采样 + 一致性投票检测幻觉，核心是 '不确定的模型会产生不一致的回答'
   - 参考：[SelfCheckGPT 论文](https://arxiv.org/abs/2303.17651)
3. 思考：如何在 RAG 系统中集成幻觉检测？
   - 思路提示：在 RAG 的 retrieve 和 generate 之间加一层 fact-checking：用检索到的源文档做 entailment 校验
   - 参考：[RAGAS — RAG 评估框架](https://arxiv.org/abs/2311.09120)


---

## 明日预告

**Day 6：开源模型生态与安全评估**
🔵 LLM 核心原理与安全基础 · 第 1 周